# Diabetes Glucose Regression
The project predicts the continuous `Glucose` value in the Pima Indians Diabetes dataset using multiple regression models.


## 1. Imports

The project uses pandas and NumPy for data handling, seaborn and matplotlib for visualization, and scikit-learn for preprocessing, imputation, regression models, and evaluation metrics.


In [ ]:
from __future__ import annotations
import argparse
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.linear_model import BayesianRidge, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
TARGET_COLUMN = "Glucose"
CLASSIFICATION_COLUMN = "Outcome"
ZERO_AS_MISSING_COLUMNS = [
    "BloodPressure",
    "BMI",
    "Glucose",
    "Insulin",
    "SkinThickness",
]


## 2. Load Dataset

The dataset is included in this repository as `data/pima_diabetes.csv`. The original classification target `Outcome` is dropped because this mini project treats the task as regression and predicts `Glucose`.


In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the diabetes dataset from CSV."""

    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")
    return pd.read_csv(data_path)


In [ ]:
data_path = Path("data") / "pima_diabetes.csv"
raw_df = load_dataset(data_path)
print("Dataset shape:", raw_df.shape)
raw_df.head()


## 3. Initial Data Inspection

Inspect feature types, missing values, zero counts, distributions, and correlations before preprocessing. Some zero values in medical measurement columns are treated as missing values.


In [ ]:
raw_df.info()


In [ ]:
raw_df.isna().sum()


In [ ]:
zero_counts = (raw_df == 0).sum()
zero_counts


In [ ]:
raw_df.describe()


In [ ]:
numeric_cols = raw_df.select_dtypes(include=["float64", "int64"]).columns
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
plt.figure(figsize=(n_cols * 5, n_rows * 4))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(raw_df[col], kde=True, bins=35)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()


## 4. Preprocessing

The preprocessing drops `Outcome`, converts invalid zero measurements to missing values, uses scaled KNN imputation, detects outliers with the IQR rule, and imputes outlier positions again.


In [ ]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean missing medical measurements and impute outliers."""

    working_df = df.copy()
    if CLASSIFICATION_COLUMN in working_df.columns:
        working_df = working_df.drop(columns=[CLASSIFICATION_COLUMN])

    for column in ZERO_AS_MISSING_COLUMNS:
        working_df[column] = working_df[column].astype(float)
        working_df[column] = working_df[column].replace(0, np.nan)

    scaler = StandardScaler()
    scaled_df = pd.DataFrame(scaler.fit_transform(working_df), columns=working_df.columns)

    imputer = KNNImputer(n_neighbors=5)
    imputed_scaled = pd.DataFrame(
        imputer.fit_transform(scaled_df),
        columns=working_df.columns,
    )
    imputed_df = pd.DataFrame(
        scaler.inverse_transform(imputed_scaled),
        columns=working_df.columns,
    )

    # Mark extreme values as missing, then impute them with KNN.
    outlier_df = imputed_df.copy()
    for column in outlier_df.columns:
        q1 = outlier_df[column].quantile(0.25)
        q3 = outlier_df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (outlier_df[column] < lower_bound) | (outlier_df[column] > upper_bound)
        outlier_df.loc[mask, column] = np.nan

    outlier_scaled = pd.DataFrame(
        scaler.fit_transform(outlier_df),
        columns=outlier_df.columns,
    )
    outlier_imputed_scaled = pd.DataFrame(
        imputer.fit_transform(outlier_scaled),
        columns=outlier_df.columns,
    )
    final_df = pd.DataFrame(
        scaler.inverse_transform(outlier_imputed_scaled),
        columns=outlier_df.columns,
    )
    return final_df


In [ ]:
clean_df = preprocess_data(raw_df)
print(clean_df.isna().sum())
clean_df.head()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y=clean_df["Glucose"])
plt.title("Glucose Distribution After Cleaning")
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(clean_df.corr(), annot=True, cmap="RdYlGn")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 5. Train/Test Split

The target is `Glucose`. All other cleaned variables are used as predictors. The split is 80 percent training and 20 percent testing with a fixed random seed.


In [ ]:
def split_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Split features and target into train/test sets."""

    features = df.drop(columns=[TARGET_COLUMN])
    target = df[TARGET_COLUMN]
    return train_test_split(
        features,
        target,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )


In [ ]:
X_train, X_test, y_train, y_test = split_data(clean_df)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 6. Regression Models

The project compares linear, regularized, nonlinear, tree-based, neighbor-based, and Bayesian regression models.


In [ ]:
def build_models() -> dict[str, object]:
    """Create the regression models used in the project."""

    return {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=0.1),
        "Polynomial Regression": (
            PolynomialFeatures(degree=2),
            LinearRegression(),
        ),
        "KNN Regression": KNeighborsRegressor(n_neighbors=5),
        "SVR": SVR(kernel="rbf"),
        "Decision Tree Regression": DecisionTreeRegressor(
            max_depth=2,
            random_state=RANDOM_STATE,
        ),
        "Random Forest Regression": RandomForestRegressor(
            n_estimators=100,
            random_state=RANDOM_STATE,
        ),
        "Bayesian Ridge Regression": BayesianRidge(),
    }

def fit_predict_model(
    model: object,
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
) -> np.ndarray:
    """Fit one model and return predictions."""

    if isinstance(model, tuple):
        transformer, regressor = model
        x_train_poly = transformer.fit_transform(x_train)
        x_test_poly = transformer.transform(x_test)
        regressor.fit(x_train_poly, y_train)
        return regressor.predict(x_test_poly)

    model.fit(x_train, y_train)
    return model.predict(x_test)

def save_prediction_plot(
    y_true: pd.Series,
    y_pred: np.ndarray,
    model_name: str,
    results_dir: Path,
) -> None:
    """Save an actual-vs-predicted scatter plot."""

    plt.figure(figsize=(7, 5))
    plt.scatter(y_true, y_pred, alpha=0.75)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--")
    plt.xlabel("Actual Glucose")
    plt.ylabel("Predicted Glucose")
    plt.title(f"{model_name}: Actual vs Predicted")
    plt.tight_layout()
    filename = model_name.lower().replace(" ", "_").replace("/", "_")
    plt.savefig(results_dir / f"{filename}_actual_vs_predicted.png", dpi=160)
    plt.close()

def evaluate_models(
    models: dict[str, object],
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    results_dir: Path,
    save_plots: bool = True,
) -> dict[str, dict[str, float]]:
    """Train and evaluate all regression models."""

    results = {}
    for model_name, model in models.items():
        predictions = fit_predict_model(model, x_train, x_test, y_train)
        mse = mean_squared_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        results[model_name] = {"mse": float(mse), "r2": float(r2)}

        if save_plots:
            save_prediction_plot(y_test, predictions, model_name, results_dir)

    return results

def save_comparison_plot(results: dict[str, dict[str, float]], results_dir: Path) -> None:
    """Save side-by-side MSE and R2 comparison charts."""

    results_df = pd.DataFrame.from_dict(results, orient="index").reset_index()
    results_df = results_df.rename(columns={"index": "model"})

    plt.figure(figsize=(12, 6))
    sns.barplot(data=results_df, x="model", y="mse", color="#4f8fd6")
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Mean Squared Error")
    plt.title("Regression Model MSE Comparison")
    plt.tight_layout()
    plt.savefig(results_dir / "mse_comparison.png", dpi=160)
    plt.close()

    plt.figure(figsize=(12, 6))
    sns.barplot(data=results_df, x="model", y="r2", color="#55a868")
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("R2 Score")
    plt.title("Regression Model R2 Comparison")
    plt.tight_layout()
    plt.savefig(results_dir / "r2_comparison.png", dpi=160)
    plt.close()

def save_feature_importance(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    results_dir: Path,
) -> pd.DataFrame:
    """Train Random Forest and save feature-importance values."""

    forest = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    forest.fit(x_train, y_train)
    importance_df = (
        pd.DataFrame({"feature": x_train.columns, "importance": forest.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    importance_df.to_csv(results_dir / "feature_importance.csv", index=False)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=importance_df, x="importance", y="feature", color="#55a868")
    plt.title("Random Forest Feature Importance")
    plt.tight_layout()
    plt.savefig(results_dir / "feature_importance.png", dpi=160)
    plt.close()
    return importance_df

def print_summary(results: dict[str, dict[str, float]]) -> None:
    """Print a compact model comparison table."""

    summary = (
        pd.DataFrame.from_dict(results, orient="index")
        .reset_index()
        .rename(columns={"index": "model"})
        .sort_values("mse")
    )
    print(summary.to_string(index=False))


## 7. Train and Evaluate Models

Each model is trained on the training set and evaluated on the test set using MSE and R2 score.


In [ ]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

models = build_models()
results = evaluate_models(
    models, X_train, X_test, y_train, y_test, results_dir, save_plots=True
)
print_summary(results)


## 8. Save Generated Results

When the notebook is run, generated metrics and plots are saved in the `results/` folder.


In [ ]:
feature_importance = save_feature_importance(X_train, y_train, results_dir)
save_comparison_plot(results, results_dir)

output = {
    "dataset_shape": list(raw_df.shape),
    "target": TARGET_COLUMN,
    "results": results,
    "feature_importance": feature_importance.to_dict(orient="records"),
}
with (results_dir / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(output, file, indent=2)

print(f"Saved generated results to: {results_dir}")


## 9. Final Recorded Results

The table below is copied from the original submitted notebook outputs.

| Model | MSE | R2 |
|---|---:|---:|
| Linear Regression | 551.3952 | 0.4521 |
| Ridge Regression | 551.4030 | 0.4521 |
| Lasso Regression | 551.6126 | 0.4518 |
| Polynomial Regression | 537.3451 | 0.4660 |
| KNN Regression | **516.3685** | **0.4869** |
| SVR | 535.1923 | 0.4682 |
| Decision Tree Regression | 532.4813 | 0.4709 |
| Random Forest Regression | 556.1475 | 0.4473 |
| Bayesian Ridge Regression | 547.9070 | 0.4555 |

The best recorded model was KNN Regression, with the lowest MSE and highest R2 score in the original run.
